In [1]:
# =============================================================================
# Credit Card Fraud Detection - Logistic Regression
# Leakage-free Pipeline (PCA features + Time/Amount preprocessing)
# =============================================================================

# -----------------------------------------------------------------------------
# 1. Import Libraries
# -----------------------------------------------------------------------------
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

In [2]:
# -----------------------------------------------------------------------------
# 2. Load Dataset
# -----------------------------------------------------------------------------
# Use the local path of the provided file
df = pd.read_csv(r"C:\Users\rajat\OneDrive\Desktop\creditcard.csv")

print("Dataset shape:", df.shape)
print(df.head())


Dataset shape: (284807, 31)
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

    

In [ ]:
# -----------------------------------------------------------------------------
# 3. Target Imbalance Check
# -----------------------------------------------------------------------------
print("\nClass distribution (%):")
print(df["Class"].value_counts(normalize=True) * 100)

# Observation: ~99.83% legitimate (0) vs ~0.17% fraud (1)
# → Use stratified split, class_weight="balanced", and focus on Recall / PR-AUC


In [3]:
# -----------------------------------------------------------------------------
# 4. Basic Data Cleaning
# -----------------------------------------------------------------------------
print("\nBasic info:")
df.info()

# Remove exact duplicate transactions
df = df.drop_duplicates()
print("\nShape after dropping duplicates:", df.shape)



Basic info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 2

In [4]:
# -----------------------------------------------------------------------------
# 5. Separate Features and Target
# -----------------------------------------------------------------------------
X = df.drop(columns="Class")
y = df["Class"]


In [5]:
# -----------------------------------------------------------------------------
# 6. Train-Test Split (before any preprocessing → no leakage)
# -----------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(f"\nTrain shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train fraud rate: {y_train.mean()*100:.4f}%")
print(f"Test  fraud rate: {y_test.mean()*100:.4f}%")



Train shape: (226980, 30), Test shape: (56746, 30)
Train fraud rate: 0.1665%
Test  fraud rate: 0.1674%


In [6]:
# -----------------------------------------------------------------------------
# 7. Separate PCA and Non-PCA Features
# -----------------------------------------------------------------------------
# V1–V28 are already PCA-transformed (and standardized)
# Time and Amount were NOT included in the original PCA

pca_features = X_train.iloc[:, 1:29]          # V1 to V28
non_pca_features = X_train[["Time", "Amount"]]


In [7]:
# -----------------------------------------------------------------------------
# 8. Diagnostic Analysis – PCA Features
# -----------------------------------------------------------------------------
diagnosis_pca = {}
for column in pca_features.columns:
    diagnosis_pca[column] = {
        "Missing Values": pca_features[column].isnull().sum(),
        "Variance": pca_features[column].var(),
        "Standard Deviation": pca_features[column].std(),
        "Range": pca_features[column].max() - pca_features[column].min(),
        "Infinite Values": pca_features[column].isin([np.inf, -np.inf]).sum()
    }

diagnosis_pca = pd.DataFrame.from_dict(diagnosis_pca, orient="index")
print("\nPCA Feature Diagnosis:")
print(diagnosis_pca)

# Summary: No missing/infinite values. Already standardized → no further preprocessing needed.



PCA Feature Diagnosis:
     Missing Values  Variance  Standard Deviation      Range  Infinite Values
V1                0  3.791951            1.947293  58.862440                0
V2                0  2.721236            1.649617  94.773457                0
V3                0  2.271863            1.507270  43.063542                0
V4                0  2.007776            1.416960  22.558515                0
V5                0  1.848628            1.359643  66.893795                0
V6                0  1.751389            1.323401  47.553575                0
V7                0  1.440434            1.200181  77.860418                0
V8                0  1.367362            1.169342  93.223927                0
V9                0  1.196956            1.094055  29.029061                0
V10               0  1.151784            1.073212  48.333399                0
V11               0  1.037672            1.018662  16.816387                0
V12               0  0.984790           

In [8]:
# -----------------------------------------------------------------------------
# 9. Diagnostic Analysis – Non-PCA Features (Time & Amount)
# -----------------------------------------------------------------------------
diagnosis_non_pca = {}
for column in non_pca_features.columns:
    Q1 = non_pca_features[column].quantile(0.25)
    Q3 = non_pca_features[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((non_pca_features[column] < lower) | (non_pca_features[column] > upper)).sum()

    diagnosis_non_pca[column] = {
        "Missing Values": non_pca_features[column].isnull().sum(),
        "Outliers": outliers,
        "Skewness": non_pca_features[column].skew(),
        "Infinite Values": non_pca_features[column].isin([np.inf, -np.inf]).sum(),
        "Variance": non_pca_features[column].var(),
        "Standard Deviation": non_pca_features[column].std(),
        "Range": non_pca_features[column].max() - non_pca_features[column].min()
    }

diagnosis_non_pca = pd.DataFrame.from_dict(diagnosis_non_pca, orient="index")
print("\nNon-PCA Feature Diagnosis:")
print(diagnosis_non_pca)

# Summary:
# - No missing values
# - Significant outliers → IQR clipping
# - High skewness (especially Amount) → Yeo-Johnson power transform
# - Median imputation kept for pipeline robustness



Non-PCA Feature Diagnosis:
        Missing Values  Outliers   Skewness  Infinite Values      Variance  \
Time                 0         0  -0.037474                0  2.255130e+09   
Amount               0     25316  14.393212                0  6.040144e+04   

        Standard Deviation      Range  
Time          47488.214282  172792.00  
Amount          245.767046   19656.53  


In [9]:
# -----------------------------------------------------------------------------
# 10. Custom Transformer – IQR Outlier Clipper
# -----------------------------------------------------------------------------
class IQRClipper(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.q1_ = np.percentile(X, 25, axis=0)
        self.q3_ = np.percentile(X, 75, axis=0)
        self.iqr_ = self.q3_ - self.q1_
        return self

    def transform(self, X):
        lower = self.q1_ - 1.5 * self.iqr_
        upper = self.q3_ + 1.5 * self.iqr_
        return np.clip(X, lower, upper)


In [11]:

# -----------------------------------------------------------------------------
# 11. Build Preprocessing Pipeline
# -----------------------------------------------------------------------------
# Only Time and Amount need preprocessing
non_pca_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("outlier_clipping", IQRClipper()),
    ("power_transform", PowerTransformer(method="yeo-johnson"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("non_pca", non_pca_pipeline, ["Time", "Amount"])
    ],
    remainder="passthrough"   # V1–V28 pass through unchanged
)

In [12]:
# -----------------------------------------------------------------------------
# 12. Full Modeling Pipeline
# -----------------------------------------------------------------------------
pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", LogisticRegression())
])

In [13]:
# -----------------------------------------------------------------------------
# 13. Hyperparameter Grid
# -----------------------------------------------------------------------------
param_grid = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__penalty": ["l2"],
    "model__solver": ["liblinear"],
    "model__class_weight": ["balanced"],   # critical for imbalance
    "model__max_iter": [500]
}

In [14]:
# -----------------------------------------------------------------------------
# 14. Stratified Cross-Validation
# -----------------------------------------------------------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [15]:
# -----------------------------------------------------------------------------
# 15. Hyperparameter Tuning (optimize Average Precision / PR-AUC)
# -----------------------------------------------------------------------------
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="average_precision",   # best for highly imbalanced data
    cv=cv,
    n_jobs=-1,
    verbose=1
)

print("\nStarting GridSearchCV...")
grid_search.fit(X_train, y_train)

print("\nBest Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation Average Precision (PR-AUC):")
print(f"{grid_search.best_score_:.6f}")



Starting GridSearchCV...
Fitting 5 folds for each of 4 candidates, totalling 20 fits

Best Parameters:
{'model__C': 1, 'model__class_weight': 'balanced', 'model__max_iter': 500, 'model__penalty': 'l2', 'model__solver': 'liblinear'}

Best Cross-Validation Average Precision (PR-AUC):
0.755663


In [16]:
# -----------------------------------------------------------------------------
# 16. Predictions on Unseen Test Set
# -----------------------------------------------------------------------------
y_pred = grid_search.predict(X_test)
y_prob = grid_search.predict_proba(X_test)[:, 1]   # probability of fraud


In [17]:
# -----------------------------------------------------------------------------
# 17. Model Evaluation
# -----------------------------------------------------------------------------
print("\n" + "="*60)
print("MODEL PERFORMANCE ON TEST SET")
print("="*60)

print(f"Accuracy          : {accuracy_score(y_test, y_pred):.4f}")
print(f"Balanced Accuracy : {balanced_accuracy_score(y_test, y_pred):.4f}")
print(f"Precision         : {precision_score(y_test, y_pred):.4f}")
print(f"Recall            : {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score          : {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC           : {roc_auc_score(y_test, y_prob):.4f}")
print(f"PR-AUC            : {average_precision_score(y_test, y_prob):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))


MODEL PERFORMANCE ON TEST SET
Accuracy          : 0.9747
Balanced Accuracy : 0.9243
Precision         : 0.0551
Recall            : 0.8737
F1 Score          : 0.1037
ROC-AUC           : 0.9620
PR-AUC            : 0.6705

Confusion Matrix:
[[55229  1422]
 [   12    83]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9998    0.9749    0.9872     56651
           1     0.0551    0.8737    0.1037        95

    accuracy                         0.9747     56746
   macro avg     0.5275    0.9243    0.5455     56746
weighted avg     0.9982    0.9747    0.9857     56746



In [ ]:
# -----------------------------------------------------------------------------
# 18. Interpretation Notes (for reference)
# -----------------------------------------------------------------------------
"""
Key takeaways from this run:
- High Recall (~0.87) → most frauds are caught (only ~12 missed)
- Low Precision (~0.055) → many false alarms (legitimate tx flagged as fraud)
- Strong ranking ability (ROC-AUC ~0.96, PR-AUC ~0.67)
- This is a typical and acceptable trade-off in fraud detection when
  the cost of missing a fraud is much higher than investigating a false positive.

Possible next steps:
1. Threshold tuning (instead of default 0.5)
2. Try tree-based models (RandomForest, XGBoost, LightGBM)
3. Experiment with SMOTE / ADASYN / undersampling
4. Feature engineering on Time (hour-of-day, etc.)
"""